## Workflow for Neosurf-on-Neosurf MaSIF search

In [1]:
import os
import pandas as pd
import sys
import numpy as np
import subprocess

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)


# ----- Source code and scripts ------
# Add source_dir to python PATH
source_dir = os.path.join(repo_root, 'masif_seed_search/source')
sys.path.insert(0, source_dir)

# Add python_scripts_dir to python PATH
python_scripts_dir = os.path.join(repo_root, 'scripts/python')
sys.path.insert(0, python_scripts_dir)

prepare_input_py = os.path.join(repo_root, 'scripts/python/prepare_input.py')

# Settings:
N_ARRAY_JOBS = 500
EVOEF2_BIN = os.path.join(repo_root, "EvoEF2/EvoEF2")
N_SEED = 2000


# ----- Directories ------
data_dir = os.path.join(repo_root, 'data')

# ----- Input ------
# .csv about the seed complexes - use all human reference proteome liganded pdbs
seed_list_csv = os.path.join(data_dir, "human_reference_proteome_liganded_pdbs/human_reference_proteome_pdb_ligands_split.csv")

# Directory to store input .pdb and .sdf
input_dir = os.path.join(data_dir, 'input')
os.makedirs(input_dir, exist_ok=True)

# ----- Processing directory ------
processing_dir = os.path.join(data_dir, 'processing')
os.makedirs(processing_dir, exist_ok=True)

# 1. Prepare input files
prep_input_proc_dir = os.path.join(processing_dir, '1_prep_input')
os.makedirs(prep_input_proc_dir, exist_ok=True)

# 2. Run MaSIF preprocessing
masif_preprocess_proc_dir = os.path.join(processing_dir, '2_masif_preprocess')
os.makedirs(masif_preprocess_proc_dir, exist_ok=True)

# 3. Run MaSIF search
masif_search_proc_dir = os.path.join(processing_dir, '3_masif_search')
os.makedirs(masif_search_proc_dir, exist_ok=True)

# ----- Output ------
# Directory to write preprocessing files
preprocess_dir = os.path.join(data_dir, 'preprocess')

# Directory to write masif-search output
masif_search_out_dir = os.path.join(data_dir, 'masif_search')
os.makedirs(masif_search_out_dir, exist_ok=True)

master_subset_dir = os.path.join(masif_search_out_dir, 'subset')
os.makedirs(master_subset_dir, exist_ok=True)

query_targets_list = os.path.join(masif_search_out_dir, 'query_targets.txt')



___
### Step 1 - preprocess all targets
1. Preprocess targets with ligands in nico_targets.csv
2. Preprocess VHL and CRBN with ligands

In [2]:
df_seed = pd.read_csv(seed_list_csv)

# Use only the first N_SEED rows for testing 
if N_SEED is not None:
    df_seed = df_seed.head(N_SEED)

df_seed.head()

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,split
0,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,OFN,"~{S}-[2-[3-[[(2~{R})-4-[[[(2~{R},3~{S},4~{R},5...",CCCCCCCCCCCCCCCCCC(=O)CC(=O)SCCNC(=O)CCNC(=O)[...,0.213523,C41H72N7O18P3S,1075.386739,0.024703,41.0,25.0,1.0,seed
1,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,37X,Octyl Glucose Neopentyl Glycol,CCCCCCC(CCCCCC)(CO[C@@H]1[C@H]([C@@H]([C@H]([C...,0.213523,C27H52O12,568.345877,0.099222,27.0,12.0,3.0,seed
2,A0FGR8,ESYT2,Extended synaptotagmin-2,4P42,A,A,EGC,"2-(2-{2-[2-(2-{2-[2-(2-{2-[4-(1,1,3,3-TETRAMET...",CC(C)(C)CC(C)(C)c1ccc(cc1)OCCOCCOCCOCCOCCOCCOC...,1.000000,C32H58O10,602.402998,0.135862,32.0,10.0,1.0,seed
3,A4D1P6,WDR91,WD repeat-containing protein 91,8SHJ,A,A,ZI8,N-[3-(4-chlorophenyl)oxetan-3-yl]-4-[(3S)-3-hy...,c1cc(ccc1C(=O)NC2(COC2)c3ccc(cc3)Cl)N4CC[C@@H]...,1.000000,C20H21ClN2O3,372.124070,0.865528,20.0,5.0,1.0,seed
4,A4D1P6,WDR91,WD repeat-containing protein 91,8T55,C,C,ZI3,N-[3-(4-chlorophenyl)oxetan-3-yl]-1-propanoyl-...,CCC(=O)N1CCCc2c1cccc2C(=O)NC3(COC3)c4ccc(cc4)Cl,1.000000,C22H23ClN2O3,398.139720,0.854092,22.0,5.0,1.0,seed


In [3]:
# Prepare input files for a single complex
df_input_subset = pd.DataFrame(
    [
        {
            "pdb_id": "8VLB",
            "protein_chain": "A",
            "ligand_chain": "A",
            "ligand_code": "3JF"
        }
    ]
)

df_input_subset.to_csv(os.path.join(data_dir, f"prepare_input.csv"), index=False)

cmd = [
    "python",
    prepare_input_py,
    "--input_csv", os.path.join(data_dir, f"prepare_input.csv"),
    "--outdir", input_dir,
    "--out_csv", os.path.join(data_dir, f"prepare_input_out.csv"),
    "--evoef2_bin", EVOEF2_BIN
]
print(cmd)
# subprocess.run(cmd)


['python', '/scratch/ymeng/Neosurf_Neosurf/scripts/python/prepare_input.py', '--input_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input.csv', '--outdir', '/scratch/ymeng/Neosurf_Neosurf/data/input', '--out_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv', '--evoef2_bin', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2']


Preparing structures: 100%|██████████| 1/1 [00:02<00:00,  2.93s/it]


Wrote /scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv  (1/1 rows succeeded)


CompletedProcess(args=['python', '/scratch/ymeng/Neosurf_Neosurf/scripts/python/prepare_input.py', '--input_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input.csv', '--outdir', '/scratch/ymeng/Neosurf_Neosurf/data/input', '--out_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv', '--evoef2_bin', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2'], returncode=0)

In [ ]:
# Prepare input files
input_subset_dir = os.path.join(prep_input_proc_dir, "input_subsets")
os.makedirs(input_subset_dir, exist_ok=True)
output_subset_dir = os.path.join(prep_input_proc_dir, "output_subsets")
os.makedirs(output_subset_dir, exist_ok=True)

# Split df_seed into N_ARRAY_JOBS chunks
df_seed_subsets = np.array_split(df_seed, N_ARRAY_JOBS)

# Write each subset to a separate file
for i, df_seed_subset in enumerate(df_seed_subsets):
    df_seed_subset.to_csv(os.path.join(input_subset_dir, f"input_{i+1}.csv"), index=False)

# Submit slurm array job: task k reads input_k.csv, writes output_k.csv
cmd = [
    "sbatch",
    f"--array=1-{N_ARRAY_JOBS}",
    "scripts/slurm/prepare_input_array.sh",
    input_subset_dir,
    input_dir,
    output_subset_dir,
    EVOEF2_BIN,
]
print(cmd)
# subprocess.run(cmd, check=True)

/home/ymeng/miniconda3/envs/MaSIF/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


['sbatch', '--array=1-500', 'scripts/slurm/prepare_input_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/input', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/output_subsets', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2']
Submitted batch job 60679214


sbatch: [ESTIMATION] The estimated cost of this job is CHF 11.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 116.05      │ 11.0        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 131.3       │ 11.0        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


CompletedProcess(args=['sbatch', '--array=1-500', 'scripts/slurm/prepare_input_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/input', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/output_subsets', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2'], returncode=0)

In [5]:
# Gather all output .csv files into a single df_input_prepared
df_input_prepared = pd.DataFrame()
for i in range(N_ARRAY_JOBS):
    csv_path = os.path.join(output_subset_dir, f"output_{i+1}.csv")
    if os.path.exists(csv_path):
        df_input_prepared = pd.concat([df_input_prepared, pd.read_csv(csv_path)])
    else:
        print(f"Warning: {csv_path} does not exist")

print(f"df_input_prepared.shape: {df_input_prepared.shape}")

# Split into success and failed rows
df_preprocess_manifest = df_input_prepared[
    df_input_prepared['pdb_path'].notna() & df_input_prepared['ligand_path'].notna()
]
df_input_failed = df_input_prepared[
    df_input_prepared['pdb_path'].isna() | df_input_prepared['ligand_path'].isna()
]
print(f"Successfully prepared {df_preprocess_manifest.shape[0]} complexes.")
print(f"Failed to prepare input files for {df_input_failed.shape[0]} complexes.")
print(f"Failed entries:")
df_input_failed.head()


df_input_prepared.shape: (1980, 21)
Successfully prepared 1978 complexes.
Failed to prepare input files for 2 complexes.
Failed entries:


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,...,mw,qed,num_carbon,num_N_O,uniprot_id_count,split,pdb_path,target,ligand,ligand_path
1,O15541,RNF113A,E3 ubiquitin-protein ligase RNF113A,7DVQ,F,F,G5J,5'-O-[(S)-hydroxy{[(R)-hydroxy{[(S)-hydroxy(me...,COP(=O)(O)OP(=O)(O)OP(=O)(O)OC[C@@H]1[C@H]([C@...,1.0,...,537.006310,0.175701,11.0,19.0,3.0,seed,NaN,NaN,NaN,NaN
2,O60885,BRD4,Bromodomain-containing protein 4,6SA2,A,A,L25,~{N}-(2-methoxy-5-morpholin-4-ylsulfonyl-pheny...,Cc1c2c(c([nH]1)C(=O)Nc3cc(ccc3OC)S(=O)(=O)N4CC...,1.0,...,461.162057,0.660917,22.0,9.0,1.0,seed,NaN,NaN,NaN,NaN


In [ ]:
# MaSIF preprocess: split manifest into subsets and submit array job
preprocess_input_subset_dir = os.path.join(masif_preprocess_proc_dir, "input_subsets")
os.makedirs(preprocess_input_subset_dir, exist_ok=True)
preprocess_output_subset_dir = os.path.join(masif_preprocess_proc_dir, "output_subsets")
os.makedirs(preprocess_output_subset_dir, exist_ok=True)

df_preprocess_subsets = np.array_split(df_preprocess_manifest, N_ARRAY_JOBS)
for i, df_subset in enumerate(df_preprocess_subsets):
    df_subset.to_csv(
        os.path.join(preprocess_input_subset_dir, f"input_{i+1}.csv"),
        index=False,
    )

cmd = [
    "sbatch",
    f"--array=1-{N_ARRAY_JOBS}",
    "scripts/slurm/preprocess_array.sh",
    preprocess_input_subset_dir,
    preprocess_output_subset_dir,
]
print(cmd)
subprocess.run(cmd, check=True)

['sbatch', '--array=1-500', 'scripts/slurm/preprocess_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/output_subsets']


sbatch: [ESTIMATION] The estimated cost of this job is CHF 44.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│


CompletedProcess(args=['sbatch', '--array=1-500', 'scripts/slurm/preprocess_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/output_subsets'], returncode=0)

Submitted batch job 60769759


sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 117.6       │ 44.0        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 132.9       │ 44.0        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


In [9]:
# Gather preprocess output subsets
df_preprocess_results = pd.DataFrame()
for i in range(N_ARRAY_JOBS):
    csv_path = os.path.join(preprocess_output_subset_dir, f"output_{i+1}.csv")
    if os.path.exists(csv_path):
        df_preprocess_results = pd.concat([df_preprocess_results, pd.read_csv(csv_path)])
    else:
        print(f"Warning: {csv_path} does not exist")

print(f"df_preprocess_results.shape: {df_preprocess_results.shape}")

df_preprocess_ok = df_preprocess_results[
    df_preprocess_results["status"].isin(["success", "skipped"])
]
df_preprocess_failed = df_preprocess_results[df_preprocess_results["status"] == "error"]

print(f"Preprocessed successfully or skipped: {df_preprocess_ok.shape[0]}")
print(f"Preprocess errors: {df_preprocess_failed.shape[0]}")
print("Failed entries:")
df_preprocess_failed.head()

df_preprocess_results.shape: (1978, 23)
Preprocessed successfully or skipped: 1239
Preprocess errors: 739
Failed entries:


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,...,num_carbon,num_N_O,uniprot_id_count,split,pdb_path,target,ligand,ligand_path,status,error_message
0,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,OFN,"~{S}-[2-[3-[[(2~{R})-4-[[[(2~{R},3~{S},4~{R},5...",CCCCCCCCCCCCCCCCCC(=O)CC(=O)SCCNC(=O)CCNC(=O)[...,0.213523,...,41.0,25.0,1.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/6Y7F...,6Y7F-OFN_A,OFN_A,/scratch/ymeng/Neosurf_Neosurf/data/input/6Y7F...,error,/usr/lib/x86_64-linux-gnu/libstdc++.so.6: vers...
1,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,37X,Octyl Glucose Neopentyl Glycol,CCCCCCC(CCCCCC)(CO[C@@H]1[C@H]([C@@H]([C@H]([C...,0.213523,...,27.0,12.0,3.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/6Y7F...,6Y7F-37X_A,37X_A,/scratch/ymeng/Neosurf_Neosurf/data/input/6Y7F...,error,/usr/lib/x86_64-linux-gnu/libstdc++.so.6: vers...
3,A4D1P6,WDR91,WD repeat-containing protein 91,8SHJ,A,A,ZI8,N-[3-(4-chlorophenyl)oxetan-3-yl]-4-[(3S)-3-hy...,c1cc(ccc1C(=O)NC2(COC2)c3ccc(cc3)Cl)N4CC[C@@H]...,1.000000,...,20.0,5.0,1.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/8SHJ...,8SHJ-ZI8_A,ZI8_A,/scratch/ymeng/Neosurf_Neosurf/data/input/8SHJ...,error,/usr/lib/x86_64-linux-gnu/libstdc++.so.6: vers...
1,A4D1P6,WDR91,WD repeat-containing protein 91,9DTA,A,A,A1BBV,"N-(1-benzoyl-1,2,3,4-tetrahydroquinolin-6-yl)-...",COc1c(cccc1C#N)CC(=O)Nc2ccc3c(c2)CCCN3C(=O)c4c...,1.000000,...,26.0,6.0,1.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/9DTA...,9DTA-A1BBV_A,A1B_A,/scratch/ymeng/Neosurf_Neosurf/data/input/9DTA...,error,/usr/lib/x86_64-linux-gnu/libstdc++.so.6: vers...
2,A4D1P6,WDR91,WD repeat-containing protein 91,9DTB,A,A,A1BBW,"N-[(1R)-4-cyano-2,3-dihydro-1H-inden-1-yl]-4-[...",c1ccc2c(c1)c(ncn2)Nc3ccc(cc3)C(=O)N[C@@H]4CCc5...,1.000000,...,25.0,6.0,1.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/9DTB...,9DTB-A1BBW_A,A1B_A,/scratch/ymeng/Neosurf_Neosurf/data/input/9DTB...,error,/usr/lib/x86_64-linux-gnu/libstdc++.so.6: vers...


In [13]:
# Verify seed_ids are unique (one entry per pdb_id + ligand_code combination)
assert df_preprocessed_seeds["seed_id"].is_unique, "seed_id is not unique!"
df_preprocessed_seeds["seed_id"].value_counts().head()

Need a entry id that identifies the ligand as well


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,...,num_N_O,uniprot_id_count,split,pdb_path,target,ligand,ligand_path,status,error_message,seed_id
0,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,OFN,"~{S}-[2-[3-[[(2~{R})-4-[[[(2~{R},3~{S},4~{R},5...",CCCCCCCCCCCCCCCCCC(=O)CC(=O)SCCNC(=O)CCNC(=O)[...,0.213523,...,25.0,1.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/6Y7F...,6Y7F_A,OFN_A,/scratch/ymeng/Neosurf_Neosurf/data/input/6Y7F...,skipped,NaN,6Y7F_A
1,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,37X,Octyl Glucose Neopentyl Glycol,CCCCCCC(CCCCCC)(CO[C@@H]1[C@H]([C@@H]([C@H]([C...,0.213523,...,12.0,3.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/6Y7F...,6Y7F_A,37X_A,/scratch/ymeng/Neosurf_Neosurf/data/input/6Y7F...,skipped,NaN,6Y7F_A
11,A8MTJ3,GNAT3,Guanine nucleotide-binding protein G(t) subuni...,8VY9,A,R,A1AEI,4-methyl-N-[(2M)-2-(1H-tetrazol-5-yl)phenyl]-6...,Cc1cc(nc(n1)Nc2ccccc2c3[nH]nnn3)C(F)(F)F,1.000000,...,7.0,3.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/8VY9...,8VY9_AR,A1A_R,/scratch/ymeng/Neosurf_Neosurf/data/input/8VY9...,skipped,NaN,8VY9_AR
12,A8MTJ3,GNAT3,Guanine nucleotide-binding protein G(t) subuni...,8YKY,A,R,A1AEI,4-methyl-N-[(2M)-2-(1H-tetrazol-5-yl)phenyl]-6...,Cc1cc(nc(n1)Nc2ccccc2c3[nH]nnn3)C(F)(F)F,1.000000,...,7.0,3.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/8YKY...,8YKY_AR,A1A_R,/scratch/ymeng/Neosurf_Neosurf/data/input/8YKY...,skipped,NaN,8YKY_AR
20,O00206,TLR4,Toll-like receptor 4,3FXI,B,A,FTT,3-HYDROXY-TETRADECANOIC ACID,CCCCCCCCCCC[C@H](CC(=O)O)O,0.000000,...,3.0,3.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/3FXI...,3FXI_BA,FTT_A,/scratch/ymeng/Neosurf_Neosurf/data/input/3FXI...,skipped,NaN,3FXI_BA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32,P43116,PTGER2,Prostaglandin E2 receptor EP2 subtype,7CX3,R,R,GNO,2-[3-[[(4-pyrazol-1-ylphenyl)methyl-pyridin-3-...,c1cc(cc(c1)OCC(=O)O)CN(Cc2ccc(cc2)n3cccn3)S(=O...,0.351955,...,9.0,2.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/7CX3...,7CX3_R,GNO_R,/scratch/ymeng/Neosurf_Neosurf/data/input/7CX3...,skipped,NaN,7CX3_R
33,P43116,PTGER2,Prostaglandin E2 receptor EP2 subtype,7CX4,R,R,GM9,2-[3-[[(4-~{tert}-butylphenyl)methyl-pyridin-3...,CC(C)(C)c1ccc(cc1)CN(Cc2cccc(c2)OCC(=O)O)S(=O)...,0.351955,...,7.0,2.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/7CX4...,7CX4_R,GM9_R,/scratch/ymeng/Neosurf_Neosurf/data/input/7CX4...,skipped,NaN,7CX4_R
27,Q9BRQ0,PYGO2,Pygopus homolog 2,4UP5,A,A,94W,"6-methoxy-1,3-benzothiazol-2-amine",COc1ccc2c(c1)sc(n2)N,1.000000,...,3.0,2.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/4UP5...,4UP5_A,94W_A,/scratch/ymeng/Neosurf_Neosurf/data/input/4UP5...,skipped,NaN,4UP5_A
46,A0A0B4J240,TRAV10,T cell receptor alpha variable 10,6V80,C,A,AGH,"N-{(1S,2R,3S)-1-[(ALPHA-D-GALACTOPYRANOSYLOXY)...",CCCCCCCCCCCCCCCCCCCCCCCCCC(=O)N[C@@H](CO[C@@H]...,0.000000,...,10.0,3.0,seed,/scratch/ymeng/Neosurf_Neosurf/data/input/6V80...,6V80_CA,AGH_A,/scratch/ymeng/Neosurf_Neosurf/data/input/6V80...,skipped,NaN,6V80_CA


___
### Step 2 - Run masif_search around ligand residues
Query with target around the ligand (e.g. VHL), search for seed patches around the seed ligands

In [10]:
# Get seed_ids from df_seed
from prepare_input import chain_suffix

def row_to_seed_target(row):
    pdb_id = row.pdb_id
    protein_chain = row.protein_chain
    ligand_chain = row.ligand_chain
    ligand_code = row.ligand_code
    chains = chain_suffix(protein_chain, ligand_chain)
    target = f"{pdb_id}-{ligand_code}_{chains}"
    return target

df_preprocess_ok["seed_id"] = df_preprocess_ok.apply(row_to_seed_target, axis=1)
df_preprocessed_seeds = df_preprocess_ok[df_preprocess_ok["split"] == "seed"]            # seed-ligand complexes
df_preprocessed_targets = df_preprocess_ok[df_preprocess_ok["split"] == "target"]        # known E3 ligase-ligand complexes
seed_ids = df_preprocessed_seeds["seed_id"].tolist()
seed_ids[0:5]

/tmp/ipykernel_4189633/1884664044.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_preprocess_ok["seed_id"] = df_preprocess_ok.apply(row_to_seed_target, axis=1)


['6Y7F_A', '6Y7F_A', '4P42_A', '8SHJ_A', '8T55_C']

#### Example for a single target

In [11]:
from pathlib import Path
import subprocess

def submit_neosurf_search(
    seed_ids,
    query_target,
    masif_search_out_dir,
    n_subsets=500,
    dry_run=True
):
    import shutil
    import os

    query_out_dir = os.path.join(masif_search_out_dir, query_target)
    subset_dir = Path(query_out_dir) / "subset"

    # Clear the subset directory before writing new seeds
    if subset_dir.exists():
        shutil.rmtree(subset_dir)
    subset_dir.mkdir(parents=True, exist_ok=True)

    # Evenly split all seed ids across N_SUBSET files (one masif_search.py call per file)
    chunks = [[] for _ in range(n_subsets)]
    for i, seed_id in enumerate(seed_ids):
        chunks[i % n_subsets].append(seed_id)

    for chunk_ix, chunk in enumerate(chunks, start=1):
        if chunk:
            (subset_dir / str(chunk_ix)).write_text("\n".join(chunk) + "\n")

    n_subsets_actual = sum(1 for chunk in chunks if chunk)

    # Clamp n_subsets to number of seeds
    n_subsets_to_use = min(n_subsets_actual, len(seed_ids))

    subset_dir_abs = os.path.abspath(subset_dir)
    print(f"Query target: {query_target}")
    print(f"Wrote {len(seed_ids)} seed(s) into {n_subsets_to_use} subset file(s) under {subset_dir}")
    submit_command = (
        f"sbatch --array=1-{n_subsets_to_use} scripts/slurm/search_array.sh "
        f"{query_target} {os.path.abspath(masif_search_out_dir)} {subset_dir_abs}"
    )
    print(f"Submit: {submit_command}")

    if not dry_run:
        subprocess.run(submit_command, shell=True)

# Example usage:
submit_neosurf_search(seed_ids, query_target="8VLB_A", masif_search_out_dir=masif_search_out_dir, n_subsets=N_ARRAY_JOBS, dry_run=True)

Query target: 8VLB_A
Wrote 24540 seed(s) into 500 subset file(s) under /scratch/ymeng/Neosurf_Neosurf/data/masif_search/8VLB_A/subset
Submit: sbatch --array=1-500 scripts/slurm/search_array.sh 8VLB_A /scratch/ymeng/Neosurf_Neosurf/data/masif_search /scratch/ymeng/Neosurf_Neosurf/data/masif_search/8VLB_A/subset


#### Submit array job to search using all available unique target complexes

In [ ]:
# Deduplicate target complexes to unique ligase-compound complexes
print(f"df_seed_targets.shape: {df_preprocessed_targets.shape}")
df_preprocessed_targets_dedup = df_preprocessed_targets.drop_duplicates(subset=["uniprot_id", "ligand_code"])
print(f"df_seed_targets_dedup.shape: {df_preprocessed_targets_dedup.shape}")
df_preprocessed_targets_dedup.head()

df_seed_targets.shape: (389, 24)
df_seed_targets_dedup.shape: (230, 24)


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,...,num_N_O,uniprot_id_count,split,pdb_path,target,ligand,ligand_path,status,error_message,seed_id
9,P40337,VHL,von Hippel-Lindau disease tumor suppressor,3ZRC,F,F,L8B,(4R)-4-HYDROXY-1-[(3-METHYLISOXAZOL-5-YL)ACETY...,Cc1cc(on1)CC(=O)N2C[C@@H](C[C@H]2C(=O)NCc3ccc(...,1.0,...,9.0,1.0,target,/scratch/ymeng/Neosurf_Neosurf/data/input/3ZRC...,3ZRC_F,L8B_F,/scratch/ymeng/Neosurf_Neosurf/data/input/3ZRC...,skipped,NaN,3ZRC_F
10,P40337,VHL,von Hippel-Lindau disease tumor suppressor,3ZTC,L,L,TR0,(4R)-N-(BIPHENYL-4-YLMETHYL)-4-HYDROXY-1-[(3-M...,Cc1cc(on1)CC(=O)N2C[C@@H](C[C@H]2C(=O)NCc3ccc(...,1.0,...,7.0,1.0,target,/scratch/ymeng/Neosurf_Neosurf/data/input/3ZTC...,3ZTC_L,TR0_L,/scratch/ymeng/Neosurf_Neosurf/data/input/3ZTC...,skipped,NaN,3ZTC_L
11,P40337,VHL,von Hippel-Lindau disease tumor suppressor,3ZTD,C,C,ZTD,METHYL 4-[({(4R)-4-HYDROXY-1-[(3-METHYLISOXAZO...,Cc1cc(on1)CC(=O)N2C[C@@H](C[C@H]2C(=O)NCc3ccc(...,1.0,...,9.0,1.0,target,/scratch/ymeng/Neosurf_Neosurf/data/input/3ZTD...,3ZTD_C,ZTD_C,/scratch/ymeng/Neosurf_Neosurf/data/input/3ZTD...,skipped,NaN,3ZTD_C
12,P40337,VHL,von Hippel-Lindau disease tumor suppressor,3ZUN,C,C,ZUN,"(4R)-4-hydroxy-1-[(3-methyl-1,2-oxazol-5-yl)ac...",Cc1cc(on1)CC(=O)N2C[C@@H](C[C@H]2C(=O)NCc3ccc(...,1.0,...,10.0,1.0,target,/scratch/ymeng/Neosurf_Neosurf/data/input/3ZUN...,3ZUN_C,ZUN_C,/scratch/ymeng/Neosurf_Neosurf/data/input/3ZUN...,skipped,NaN,3ZUN_C
13,P40337,VHL,von Hippel-Lindau disease tumor suppressor,4AWJ,F,F,V6F,(4R)-1-acetyl-4-hydroxy-N-methyl-L-prolinamide,CC(=O)N1C[C@@H](C[C@H]1C(=O)NC)O,1.0,...,5.0,1.0,target,/scratch/ymeng/Neosurf_Neosurf/data/input/4AWJ...,4AWJ_F,V6F_F,/scratch/ymeng/Neosurf_Neosurf/data/input/4AWJ...,skipped,NaN,4AWJ_F


In [18]:
# Write query_targets.txt for search_array.sh
with open(query_targets_list, 'w') as f:
    for target in df_preprocessed_targets_dedup["seed_id"]:
        f.write(f"{target}\n")

# Split seed_ids into N_ARRAY_JOBS chunks and write them to master_subset_dir
seed_chunks = np.array_split(seed_ids, N_ARRAY_JOBS)
os.makedirs(master_subset_dir, exist_ok=True)
for idx, chunk in enumerate(seed_chunks):
    subset_file = os.path.join(master_subset_dir, f"{idx+1}")
    with open(subset_file, "w") as sf:
        for seed in chunk:
            sf.write(f"{seed}\n")


In [21]:
# Submit array job:
cmd = [
    "sbatch",
    f"--array=1-{N_ARRAY_JOBS}",
    "scripts/slurm/search_array.sh",
    query_targets_list,
    masif_search_out_dir,
    master_subset_dir,
]
print(cmd)
subprocess.run(cmd, check=True)

['sbatch', '--array=1-500', 'scripts/slurm/search_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/masif_search/query_targets.txt', '/scratch/ymeng/Neosurf_Neosurf/data/masif_search', '/scratch/ymeng/Neosurf_Neosurf/data/masif_search/subset']
Submitted batch job 60447319


sbatch: [ESTIMATION] The estimated cost of this job is CHF 66.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 110.95      │ 66.0        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 126.25      │ 66.0        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


CompletedProcess(args=['sbatch', '--array=1-500', 'scripts/slurm/search_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/masif_search/query_targets.txt', '/scratch/ymeng/Neosurf_Neosurf/data/masif_search', '/scratch/ymeng/Neosurf_Neosurf/data/masif_search/subset'], returncode=0)

____

In [4]:
def print_pymol_commands(row):
    matched_protein = row["matched_protein"]
    matched_pdb = matched_protein.split("_")[0]
    matched_chains = matched_protein.split("_")[1]
    matched_patch_id = row["matched_patch_id"]

    target_protein = row["target"]
    target_pdb = target_protein.split("_")[0]
    target_chains = target_protein.split("_")[1]
    flattened_transform = row["flattened_transform"]

    # Split matched_chains (e.g. "AB") into ["A", "B"]
    matched_chains = list(matched_chains)
    # Joint matched_chains into a string (e.g. "chain A chain B")
    matched_chains_str = "chain " + " chain ".join(matched_chains)

    # Split target_chains (e.g. "AB") into ["A", "B"]
    target_chains = list(target_chains)
    target_chains_str = "chain " + " chain ".join(target_chains)

    # Load target protein
    print(f"fetch {target_pdb}, {target_protein}_{matched_protein}_{matched_patch_id}")
    print(f"remove {target_protein}_{matched_protein}_{matched_patch_id} AND (not {target_chains_str})")

    # load and transform matched protein
    print(f"fetch {matched_pdb}, {matched_protein}_{matched_patch_id}")
    print(f"remove {matched_protein}_{matched_patch_id} AND (not {matched_chains_str})")
    print(f"apply_transform {matched_protein}_{matched_patch_id}, '{flattened_transform}'")

    # Copy transformed matched_protein to target protein object
    print(f"copy_to {target_protein}_{matched_protein}_{matched_patch_id}, {matched_protein}_{matched_patch_id}")

    # Delete {matched_protein}_{matched_patch_id} object
    print(f"delete {matched_protein}_{matched_patch_id}")

# Apply to all rows
df_dedup.apply(print_pymol_commands, axis=1)

fetch 6H0F, 6H0F_B_7AFW_A_80
remove 6H0F_B_7AFW_A_80 AND (not chain B)
fetch 7AFW, 7AFW_A_80
remove 7AFW_A_80 AND (not chain A)
apply_transform 7AFW_A_80, '-0.7859344656941732,-0.5043491230556201,-0.35768558498636877,-3.6363662083943424,0.11463107809434332,-0.6873130377377955,0.717259021616719,-111.81487360429189,-0.6075909245281254,0.5227167016928679,0.597997088790896,33.085055390718644,0.0,0.0,0.0,1.0'
copy_to 6H0F_B_7AFW_A_80, 7AFW_A_80
delete 7AFW_A_80
fetch 6H0G, 6H0G_B_7AFW_A_51
remove 6H0G_B_7AFW_A_51 AND (not chain B)
fetch 7AFW, 7AFW_A_51
remove 7AFW_A_51 AND (not chain A)
apply_transform 7AFW_A_51, '0.8182982172028582,-0.553642071949644,-0.1544942843277068,-72.9274821077419,0.46224671737335316,0.4741008655351706,0.7493706303134404,35.656988642990825,-0.341637234504941,-0.6846231265930978,0.6438751234002663,-102.12152938615822,0.0,0.0,0.0,1.0'
copy_to 6H0G_B_7AFW_A_51, 7AFW_A_51
delete 7AFW_A_51
fetch 7LPS, 7LPS_B_7AFW_A_70
remove 7LPS_B_7AFW_A_70 AND (not chain B)
fetch 7AFW,

9     None
15    None
28    None
43    None
59    None
66    None
71    None
75    None
dtype: object

In [43]:
def print_pymol_commands(row):
    matched_protein = row["matched_protein"]
    matched_pdb = matched_protein.split("_")[0]
    matched_chains = matched_protein.split("_")[1]
    matched_patch_id = row["matched_patch_id"]
    flattened_transform = row["flattened_transform"]

    # Split matched_chains (e.g. "AB") into ["A", "B"]
    matched_chains = list(matched_chains)
    # Joint matched_chains into a string (e.g. "chain A chain B")
    matched_chains_str = "chain " + " chain ".join(matched_chains)

    print(f"fetch {matched_pdb}, {matched_protein}_{matched_patch_id}")
    #print(f"select {matched_protein}_{matched_patch_id} AND (not {matched_chains_str})")
    #print('cmd.remove("sele");cmd.delete("sele")')
    print(f"apply_transform {matched_protein}_{matched_patch_id}, '{flattened_transform}'")

# Apply to all rows
df_dedup.apply(print_pymol_commands, axis=1)

fetch 5QSQ, 5QSQ_B_40
apply_transform 5QSQ_B_40, '0.18811655710121694,0.3201757470768955,0.9284932158762048,-63.34899149991804,0.1194347315502052,-0.945812608023888,0.30195008760154796,-21.857826309349356,0.9748576849181194,0.05409252708834745,-0.21616311588539106,-29.57452413756281,0.0,0.0,0.0,1.0'
fetch 5QSV, 5QSV_D_29
apply_transform 5QSV_D_29, '-0.8848374941118574,-0.459948308188821,-0.07423047088689808,-154.06565366628317,0.4264906636583265,-0.8637764209055722,0.2683207194754801,4.137717891846782,-0.18753219143957434,0.20576163024675115,0.96046542295497,101.9424492820446,0.0,0.0,0.0,1.0'
fetch 6M92, 6M92_CA_58
apply_transform 6M92_CA_58, '0.34474053707697766,0.7252827786602123,-0.5959184953286809,-67.98240992873171,-0.5430922038973354,-0.3636905840683252,-0.7568223154254735,-60.38063982437132,-0.7656401375070488,0.5845460204628572,0.2685165354298039,-40.661634546822384,0.0,0.0,0.0,1.0'
fetch 6M92, 6M92_CA_183
apply_transform 6M92_CA_183, '-0.054155842613027305,-0.8787217692945033,

0    None
1    None
2    None
3    None
4    None
5    None
6    None
7    None
8    None
dtype: object